In [ ]:
from typing import Optional
from pydantic import BaseModel, Field

# ============================================================
# 1. DATA MODELS
# ============================================================
class Customer(BaseModel):
    id: str
    name: str
    age: int
    city: str
    status: str

class Order(BaseModel):
    order_id: str
    product: str
    amount: float
    status: str

class RiskAnalysis(BaseModel):
    risk: str
    score: float
    reasons: list[str]

class AgentState(BaseModel):
    task: str
    customer_id: str
    customer: Optional[Customer] = None
    orders: list[Order] = []
    total_spending: float = 0
    overdue_orders: int = 0
    risk_analysis: Optional[RiskAnalysis] = None
    tools_used: list[str] = []
    current_step: int = 0
    final_answer: Optional[str] = None

# ============================================================
# 2. OUR "DATABASE"
# ============================================================
customers = [
    {
        "id": "C101",
        "name": "John",
        "age": 35,
        "city": "Pune",
        "status": "active"
    },
    {
        "id": "C102",
        "name": "Mary",
        "age": 42,
        "city": "Mumbai",
        "status": "active"
    }
]

orders = [
    {
        "order_id": "O1",
        "customer_id": "C101",
        "product": "Laptop",
        "amount": 80000,
        "status": "delivered"
    },
    {
        "order_id": "O2",
        "customer_id": "C101",
        "product": "Phone",
        "amount": 50000,
        "status": "overdue"
    },
    {
        "order_id": "O3",
        "customer_id": "C101",
        "product": "Monitor",
        "amount": 20000,
        "status": "overdue"
    },
    {
        "order_id": "O4",
        "customer_id": "C102",
        "product": "Tablet",
        "amount": 40000,
        "status": "delivered"
    }
]
# ============================================================
# 3. TOOL 1 — GET CUSTOMER
# ============================================================
def get_customer(customer_id: str) -> Customer:
    for customer in customers:
        if customer["id"] == customer_id:
            return Customer(**customer)
    raise ValueError("Customer not found")

# ============================================================
# 4. TOOL 2 — GET ORDERS
# ============================================================
def get_orders(customer_id: str) -> list[Order]:
    customer_orders = [
        order
        for order in orders
        if order["customer_id"] == customer_id
    ]
    return [
        Order(
            order_id=order["order_id"],
            product=order["product"],
            amount=order["amount"],
            status=order["status"]
        )
        for order in customer_orders
    ]
# ============================================================
# 5. TOOL 3 — ANALYZE CUSTOMER
# ============================================================
def analyze_customer(
    customer: Customer,
    orders: list[Order]
) -> RiskAnalysis:

    total_spending = sum(
        order.amount
        for order in orders
    )
    overdue = [
        order
        for order in orders
        if order.status == "overdue"
    ]
    reasons = []
    score = 0
    if len(overdue) > 0:
        score += 0.4
        reasons.append(
            f"{len(overdue)} overdue orders"
        )
    if total_spending > 100000:
        score += 0.3
        reasons.append(
            "High customer spending"
        )
    if customer.status == "active":
        score += 0.1
    if score >= 0.6:
        risk = "high"
    elif score >= 0.3:
        risk = "medium"
    else:
        risk = "low"
    return RiskAnalysis(
        risk=risk,
        score=score,
        reasons=reasons
    )
# ============================================================
# 6. JSON TRANSFORMATION
# ============================================================
def flatten_customer_state(state: AgentState):
    return {
        "customer_id": state.customer_id,
        "customer_name": state.customer.name
        if state.customer else None,
        "city": state.customer.city
        if state.customer else None,
        "total_spending": state.total_spending,
        "overdue_orders": state.overdue_orders,
        "risk": state.risk_analysis.risk
        if state.risk_analysis else None,
        "risk_score": state.risk_analysis.score
        if state.risk_analysis else None
    }
# ============================================================
# 7. AGENT
# ===========================================================
def run_agent(customer_id: str):

    # -------------------------------
    # Create initial agent state
    # -------------------------------

    state = AgentState(

        task="Analyze customer",

        customer_id=customer_id
    )


    # ========================================================
    # STEP 1 — CALL CUSTOMER TOOL
    # ========================================================

    state.current_step = 1

    customer = get_customer(customer_id)

    state.customer = customer

    state.tools_used.append(
        "get_customer"
    )


    # ========================================================
    # STEP 2 — CALL ORDER TOOL
    # ========================================================

    state.current_step = 2

    customer_orders = get_orders(customer_id)

    state.orders = customer_orders

    state.tools_used.append(
        "get_orders"
    )


    # ========================================================
    # STEP 3 — PROCESS JSON DATA
    # ========================================================

    state.current_step = 3

    state.total_spending = sum(

        order.amount

        for order in state.orders
    )


    state.overdue_orders = len([

        order

        for order in state.orders

        if order.status == "overdue"
    ])


    # ========================================================
    # STEP 4 — ANALYZE CUSTOMER
    # ========================================================

    state.current_step = 4

    analysis = analyze_customer(

        state.customer,

        state.orders
    )

    state.risk_analysis = analysis

    state.tools_used.append(
        "analyze_customer"
    )


    # ========================================================
    # STEP 5 — FINAL RESPONSE
    # ========================================================

    state.current_step = 5

    state.final_answer = (

        f"Customer {state.customer.name} "

        f"has {state.risk_analysis.risk} risk. "

        f"Total spending is "

        f"${state.total_spending:,.2f}."
    )


    return state


# ============================================================
# 8. RUN THE AGENT
# ============================================================

state = run_agent("C101")


# ============================================================
# 9. DISPLAY AGENT STATE
# ============================================================

print("\n===== AGENT STATE =====")

print(state.model_dump_json(indent=2))


# ============================================================
# 10. FLATTEN JSON
# ============================================================

print("\n===== FLATTENED RESULT =====")

flat = flatten_customer_state(state)

print(flat)